# Homework Answer Module 6

by Rahma Hayuning Astuti

## Q1 Refactoring

finished on batch.py, below code of batch.py for question 1 - 3

## Q2 install pytest 

need to have __init__.py in tests folder for testing with pytest

## Q3 writing test unit
 on tests/test_batch.py. answer have 3 columns

In [ ]:
#!/usr/bin/env python
# coding: utf-8

import sys
import pickle
import os
import pandas as pd
from typing import List


def read_data(filename: str) -> pd.DataFrame:
    return pd.read_parquet(filename)


def prepare_data(df: pd.DataFrame, categorical: List[str]) -> pd.DataFrame:
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()

    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')

    return df


def main(year: int, month: int) -> None:
    input_file = f'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year:04d}-{month:02d}.parquet'
    output_file = f'output/yellow_tripdata_{year:04d}-{month:02d}.parquet'
    categorical = ['PULocationID', 'DOLocationID']

    with open('model.bin', 'rb') as f_in:
        dv, lr = pickle.load(f_in)

    df = read_data(input_file)
    df = prepare_data(df, categorical)
    df['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')

    dicts = df[categorical].to_dict(orient='records')
    X_val = dv.transform(dicts)
    y_pred = lr.predict(X_val)
    print('predicted mean duration:', y_pred.mean())

    df_result = pd.DataFrame()
    df_result['ride_id'] = df['ride_id']
    df_result['predicted_duration'] = y_pred

    # Ensure output directory exists
    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    # Save only if file doesn't exist
    if not os.path.exists(output_file):
        df_result.to_parquet(output_file, engine='pyarrow', index=False)
        print('File created:', output_file)
    else:
        print('File already exists:', output_file)


if __name__ == '__main__':
    year = int(sys.argv[1])
    month = int(sys.argv[2])
    main(year, month)


## Q4 Localstack

make docker-compose.yaml. So in cmd do

- docker-compose build
- docker-compose up

and then set private key and secret key for windows cmd 
- set AWS_ACCESS_KEY_ID=abc
- set AWS_SECRET_ACCESS_KEY=xyz

next, build the bucket 
- aws --endpoint-url=http://localhost:4566 s3 mb s3://nyc-duration

Asnwer: --endpoint-url

The code on batch.py newest and integration_test used for question 5 and 6

##  Q5 create test data

to get know the size of input file we can use 

aws --endpoint-url=http://localhost:4566 s3 ls s3://nyc-duration/in/

and the answer is 3620

## Q6 Finish the integration test 

now call python integration_test.py

answer: 36.28